# Anomaly Investigation: Refund on Failed Transactions (All PSPs)

A fourth anomaly cluster, found by extending the invariant check in `01_eda.ipynb` §6 (`has_refund` vs `refunded_amount`) to a column it wasn't yet checked against: `status`.

Standalone notebook — reloads and prepares the data independently of `01_eda.ipynb`.

In [1]:
import pandas as pd

df = pd.read_csv('/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv')

In [2]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])

## Rule and global check

**Rule:** `status == 'fail'` means the payment was refused - no money was ever collected, so `has_refund == True` on a failed transaction is a logical impossibility (per the domain definitions in `CLAUDE.md`).

In [3]:
fail_refund = df[df['has_refund'] & (df['status'] == 'fail')].copy()
print(f"has_refund=True & status=='fail': {len(fail_refund):,} rows")
print(fail_refund['psp_id'].value_counts())
print(fail_refund.groupby(fail_refund['created_at'].dt.to_period('M')).size())

has_refund=True & status=='fail': 1,299 rows
psp_id
psp_alpha    690
psp_beta     352
psp_gamma    257
Name: count, dtype: int64
created_at
2025-01     65
2025-02    104
2025-03     87
2025-04    104
2025-05     94
2025-06     99
2025-07    104
2025-08    170
2025-09    115
2025-10    134
2025-11    127
2025-12     96
Freq: M, dtype: int64


**Finding:** 1,299 rows, present in all three `psp_id` (690 alpha, 352 beta, 257 gamma), spread across every month of the year (65-134/month, no spike). Unlike `bank_id == 777` or the psp_gamma/psp_beta windows, this is **not bounded to one time window** - but it doesn't need to be: this is a hard invariant violation, so every single row is independently invalid regardless of when it occurred (unlike the `psp_alpha` fail-rate case, which was a soft, non-contradictory signal and was excluded precisely because it lacked a logical impossibility).

## Refund ratio - the generator's fingerprint

If the refund amount on these impossible transactions were random noise, the `refunded_amount / amount` ratio would be smeared across many values. Check whether it is instead concentrated on one exact number.

In [4]:
fail_refund['ratio'] = fail_refund['refunded_amount'] / fail_refund['amount']
print(fail_refund['ratio'].value_counts().head(6))
print(f"\nExact ratio == 0.5: {(fail_refund['ratio'] == 0.5).sum():,} ({(fail_refund['ratio'] == 0.5).mean()*100:.1f}%)")
print(fail_refund.groupby('psp_id')['ratio'].apply(lambda s: (s == 0.5).mean() * 100))

ratio
0.500000    1221
0.500243      12
1.200000       5
1.500000       5
1.333333       5
1.243309       5
Name: count, dtype: int64

Exact ratio == 0.5: 1,221 (94.0%)
psp_id
psp_alpha    99.420290
psp_beta     80.397727
psp_gamma    98.054475
Name: ratio, dtype: float64


**Finding:** 1,221 of 1,299 rows (94.0%) have `refunded_amount` equal to **exactly half** of `amount` - a hardcoded ratio, not noise. The signature is strongest in `psp_alpha` (99.4%) and `psp_gamma` (98.1%); weaker in `psp_beta` (80.4%), which is explained below. A constant, exact ratio is the same kind of fingerprint as the constant 5s latency (`bank_id == 777`) and the flat `+10` gap (`psp_beta`'s refund anomaly) - strong evidence of synthetic injection rather than a real data-quality accident.

## Overlap with the known psp_beta refund window

`04_anomaly_refund_psp_beta.ipynb` found 2,691 `refunded_amount > amount` rows, all inside 5-9 August, all `psp_beta`, all with a flat `+10` offset - a different mechanism than the 0.5 ratio here. Check whether the two anomalies share rows.

In [5]:
in_beta_window = (
    (fail_refund['psp_id'] == 'psp_beta') &
    (fail_refund['created_at'] >= '2025-08-05') &
    (fail_refund['created_at'] <= '2025-08-09 23:59:59')
)
non_half_ratio = fail_refund['ratio'] != 0.5

print(f"Rows inside the known psp_beta window : {in_beta_window.sum()}")
print(f"Rows with a non-0.5 ratio              : {non_half_ratio.sum()}")
print(f"Overlap (in window AND non-0.5)        : {(in_beta_window & non_half_ratio).sum()}")
print(f"Non-0.5 rows outside the window         : {(non_half_ratio & ~in_beta_window).sum()}")

Rows inside the known psp_beta window : 66
Rows with a non-0.5 ratio              : 78
Overlap (in window AND non-0.5)        : 66
Non-0.5 rows outside the window         : 12


**Finding:** all 66 rows inside the known psp_beta window have a non-0.5 ratio - they are the same 66 `fail` rows already flagged in section 10 of `04_anomaly_refund_psp_beta.ipynb`, where `refunded_amount` follows the `+10` rule instead of the `0.5x` rule. The two anomalies are mechanically distinct but happen to overlap on these 66 rows because both conditions (`status == 'fail'`, real refund) can be true at once. The remaining 12 non-0.5 rows outside the window are unexplained residual noise (0.9% of the cluster) - not worth a separate investigation at this scale.

## Profile of the full cluster

In [6]:
print(fail_refund['currency'].value_counts())
print()
print(fail_refund['order_type'].value_counts())
print()
print(fail_refund['payment_method'].value_counts())
print()
print(fail_refund['error_code'].value_counts())
print()
print(f"order_id range: {fail_refund['order_id'].min():,} - {fail_refund['order_id'].max():,} (unique: {fail_refund['order_id'].nunique():,} of {len(fail_refund):,})")

currency
USD    599
CAD    179
MXN    152
EUR    143
UAH     77
GBP     76
PLN     73
Name: count, dtype: int64

order_type
recurring    1213
first          86
Name: count, dtype: int64

payment_method
applepay     823
card         408
googlepay     68
Name: count, dtype: int64

error_code
2.01    776
3.08    437
2.12     86
Name: count, dtype: int64

order_id range: 228 - 998,993 (unique: 1,299 of 1,299)


**Finding:**
- `currency` mix roughly matches the dataset baseline - not currency-specific.
- 93.4% `order_type == recurring` (1,213 of 1,299) - concentrated in recurring billing, similar in spirit to the price-ladder tiers found in `07_anomaly_amount_outliers.ipynb`, though a distinct mechanism.
- `payment_method` is skewed toward `applepay` (63.4% of this cluster vs ~15% of the full dataset) - a disproportion worth noting as part of the fingerprint, unlike the three earlier clusters which weren't payment-method-specific.
- Only 3 distinct `error_code` values appear (`2.01`, `3.08`, `2.12`) vs 13 in the full dataset - a narrow, non-representative slice of failure reasons.
- `order_id` spans nearly the full dataset range (228 to 998,993) - scattered, not positioned at one end like `bank_id == 777`.

## Conclusions

Four distinct, confirmed anomaly types now exist in this dataset:

| Anomaly | Scope | Signature | Likely nature |
|---|---|---|---|
| `bank_id == 777` | 635 rows, 1 day | Constant 5s latency, 100% fail, ID format anomaly | Synthetically inserted block |
| psp_gamma latency incident | 5,433 rows, 6 days | Latency 1-2h, gradual degrade/recover | Real provider-side outage |
| psp_beta refund anomaly | 2,691 rows, 5 days | Flat +10 over-refund | Synthetically inserted / corrupted |
| **Refund on failed transaction (this notebook)** | **1,299 rows, all year, all PSPs** | **94% exact 0.5x refund ratio, `applepay`-skewed, 66 rows overlap the psp_beta window** | **Synthetically inserted, not time-bound** |

Unlike the first three, this cluster is **not concentrated in a time window** - it's valid as a hard-invariant violation regardless. Fold into `05_conclusions_next_steps.ipynb` and the final `is_anomaly` assembly.